In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/raw_house_data.csv')

df.head()

df.info()

df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 3400 entries, 0 to 3399
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   price           3400 non-null   str    
 1   location        3400 non-null   str    
 2   property_type   0 non-null      float64
 3   bedrooms        3254 non-null   float64
 4   bathrooms       3205 non-null   float64
 5   area            1454 non-null   str    
 6   road_access     3316 non-null   str    
 7   facing          3241 non-null   str    
 8   floor           3378 non-null   float64
 9   parking         3143 non-null   str    
 10  furnish_status  3173 non-null   str    
dtypes: float64(4), str(7)
memory usage: 292.3 KB


price                0
location             0
property_type     3400
bedrooms           146
bathrooms          195
area              1946
road_access         84
facing             159
floor               22
parking            257
furnish_status     227
dtype: int64

In [22]:
#Step 1: Drop useless column

df.drop(columns=['property_type'], inplace=True)

In [23]:
#Step 2: Handle missing numerical values

df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())
df['floor'] = df['floor'].fillna(df['floor'].median())

In [24]:
#Step 3: Handle missing categorical values

df['road_access'] = df['road_access'].fillna(df['road_access'].mode()[0])
df['facing'] = df['facing'].fillna(df['facing'].mode()[0])
df['parking'] = df['parking'].fillna(df['parking'].mode()[0])
df['furnish_status'] = df['furnish_status'].fillna(df['furnish_status'].mode()[0])

In [25]:
#Step 4: Important — area
#This has 1946 missing values (more than half).

df['area'].head(20)

0              NaN
1     1000 Sq. Ft.
2              NaN
3              NaN
4     1672 Sq. Ft.
5              NaN
6              NaN
7              NaN
8              NaN
9              NaN
10    3500 Sq. Ft.
11             NaN
12    1700 Sq. Ft.
13    1000 Sq. Ft.
14             NaN
15             NaN
16             NaN
17             NaN
18             NaN
19             NaN
Name: area, dtype: str

In [26]:
#Step 4: Clean area

df['area'] = df['area'].str.extract(r'(\d+)')
df['area'] = df['area'].astype(float)

In [27]:
#Step 5: Fill missing area

df['area'] = df['area'].fillna(df['area'].median())

In [28]:
#Step 6: Check price format 

df['price'].head(20)

0       Rs. 4.95 Cr
1        Rs. 5.3 Cr
2       Rs. 3.65 Cr
3        Rs. 5.5 Cr
4       Rs. 3.75 Cr
5          Rs. 8 Cr
6       Rs. 4 Lac/m
7       Rs. 2 Lac/m
8     Rs. 50,000 /m
9        Rs. 2.4 Cr
10      Rs. 1.75 Cr
11      Rs. 4.25 Cr
12      Rs. 2.75 Cr
13       Rs. 4.3 Cr
14    Rs. 45,000 /m
15       Rs. 2.9 Cr
16       Rs. 2.9 Cr
17      Rs. 6.85 Cr
18      Rs. 3 Lac/m
19       Rs. 6.5 Cr
Name: price, dtype: str

In [29]:
#Step 1: Remove rent rows

df = df[~df['price'].str.contains('/m', na=False)]

In [30]:
#Step 2: Convert price into numbers

# Remove rent, annual, and unsupported price rows first

df = df[~df['price'].str.contains('/m', na=False)]
df = df[~df['price'].str.contains('/y', na=False)]
df = df[~df['price'].str.contains('/aana', na=False)]
df = df[~df['price'].str.contains('Price on call', na=False)]


def convert_price(price):
    if pd.isna(price):
        return np.nan

    price = str(price).replace('Rs.', '').replace(',', '').strip()

    if not price:
        return np.nan

    price_lower = price.lower()

    if 'cr' in price_lower:
        return float(price_lower.replace('cr', '').strip()) * 10000000
    elif 'lac' in price_lower:
        return float(price_lower.replace('lac', '').strip()) * 100000
    else:
        try:
            return float(price)
        except ValueError:
            return np.nan


df['price'] = df['price'].apply(convert_price)
df['price'].head(10)

0     49500000.0
1     53000000.0
2     36500000.0
3     55000000.0
4     37500000.0
5     80000000.0
9     24000000.0
10    17500000.0
11    42500000.0
12    27500000.0
Name: price, dtype: float64

In [31]:
df.head()
df.isnull().sum()

price             0
location          0
bedrooms          0
bathrooms         0
area              0
road_access       0
facing            0
floor             0
parking           0
furnish_status    0
dtype: int64

In [32]:
#Step 7: Standardize text columns

text_cols = ['location', 'road_access', 'facing', 'parking', 'furnish_status']

for col in text_cols:
    df[col] = df[col].str.lower().str.strip()

In [33]:
#Step 8: Remove duplicates

df.drop_duplicates(inplace=True)
print(df.shape)

(2739, 10)


In [34]:
for col in ['road_access', 'facing', 'parking', 'furnish_status']:
    print("\n", col)
    print(df[col].unique())


 road_access
<StringArray>
[  '20 feet',   '13 feet',   '26 feet',   '11 feet',   '15 feet',   '21 feet',
   '14 feet',   '16 feet',   '18 feet',   '24 feet',   '4 meter',   '12 feet',
   '10 feet',    '4 feet',   '22 feet',    '7 feet',    '8 feet',   '23 feet',
   '36 feet',   '32 feet',   '30 feet',   '17 feet',   '19 feet',   '25 feet',
   '28 feet',  '11 meter',   '8 meter',    '5 feet',    '9 feet',   '40 feet',
    '6 feet', '2000 feet', '1600 feet']
Length: 33, dtype: str

 facing
<StringArray>
['north-west', 'south-west',      'south',       'east', 'north-east',
      'north',       'west', 'south-east']
Length: 8, dtype: str

 parking
<StringArray>
[  '1 car  & 2 bikes',   '1 car  & 3 bikes',  '2 cars  & 5 bikes',
  '4 cars  & 3 bikes',   '1 car  & 5 bikes',  '2 cars  & 2 bikes',
    '1 car  & 1 bike',  '2 cars  & 4 bikes',   '2 cars  & 1 bike',
  '3 cars  & 2 bikes',              '1 car',            '2 bikes',
  '3 cars  & 8 bikes',   '1 car  & 4 bikes',  '2 cars  & 6 bike

In [35]:
#cleaning more data 

df['road_access'] = df['road_access'].str.extract(r'(\d+)')
df['road_access'] = df['road_access'].astype(float)

In [36]:
#cleaning parking column 

import re

def extract_parking(parking):
    nums = re.findall(r'\d+', parking)
    return sum(map(int, nums)) if nums else 0

df['parking'] = df['parking'].apply(extract_parking)

In [37]:
df.info()
df.head()

<class 'pandas.DataFrame'>
Index: 2739 entries, 0 to 3399
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   price           2739 non-null   float64
 1   location        2739 non-null   str    
 2   bedrooms        2739 non-null   float64
 3   bathrooms       2739 non-null   float64
 4   area            2739 non-null   float64
 5   road_access     2739 non-null   float64
 6   facing          2739 non-null   str    
 7   floor           2739 non-null   float64
 8   parking         2739 non-null   int64  
 9   furnish_status  2739 non-null   str    
dtypes: float64(6), int64(1), str(3)
memory usage: 235.4 KB


,price,location,bedrooms,bathrooms,area,road_access,facing,floor,parking,furnish_status
0,49500000.0,"imadol, lalitpur",6.0,4.0,1700.0,20.0,north-west,3.0,3,full-furnished
1,53000000.0,"dhapasi, kathmandu",6.0,4.0,1000.0,13.0,south-west,3.0,3,un-furnished
2,36500000.0,"tokha, kathmandu",5.0,3.0,1700.0,26.0,south,2.5,4,full-furnished
3,55000000.0,"hattiban, lalitpur",3.0,3.0,1700.0,11.0,south,2.5,3,semi-furnished
4,37500000.0,"sanepa, lalitpur",4.0,3.0,1672.0,20.0,south,2.5,3,un-furnished


In [38]:
df.to_csv("cleaned_house_data.csv", index=False)